In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import xarray as xr
from glob import glob
import os
from netCDF4 import Dataset
import pandas as pd
from datetime import datetime, date, timedelta
from pathlib import Path
import scipy
import scipy.ndimage
from mpl_toolkits.axes_grid1 import ImageGrid
import math
import cc3d

from mpl_toolkits.axes_grid1 import make_axes_locatable

t = 0

input_path = Path("/mnt/stor-pool-01/projects/heus/ShellAnalysis/SEUS")
fname = "slab_entrainment_stats.nc"
file_path = input_path / fname

In [ ]:
# Plotting

GROUPS_STRUCTURE = {
    "Average": {
        "Domain": ["Cloud", "Shell"],
        "Shallow": ["Cloud", "Shell"],
        "Congestus": ["Cloud", "Shell"],
        "Deep": ["Cloud", "Shell"],
        "Free_shell": ["Shell"]
    },
    "Sum": {
        "Domain": ["Cloud", "Shell"],
        "Shallow": ["Cloud", "Shell"],
        "Congestus": ["Cloud", "Shell"],
        "Deep": ["Cloud", "Shell"],
        "Free_shell": ["Shell"]
    }
}

types = 1
categories = 5
bodies = 2
graph_width = categories * bodies
graph_height = types * categories

fig = plt.figure(figsize=(8 * graph_width, 12 * graph_height), layout='constrained')
if types == 1:
    sb_types = [fig.subfigures(nrows=1, ncols=1)]
else:
    sb_types = fig.subfigures(nrows=types, ncols=1)

i = 0
for top in ["Average"]: #GROUPS_STRUCTURE.keys():
    sb_types[i].suptitle(f"Method: {top}", fontsize=20, weight='bold')
    sb_cats = sb_types[i].subfigures(nrows=categories, ncols=1) # One subfigure per row
    
    j = 0
    for mid, bottoms in GROUPS_STRUCTURE[top].items():
        sb_cats[j].suptitle(f"Category: {mid}", fontsize=12, weight='bold')
        sb_body = sb_cats[j].subplots(nrows=1, ncols=bodies)

        k = 0
        for bot in bottoms:
            ax = sb_body[k]
            

            group_path = f"/{top}/{mid}/{bot}"

            try:
                # 2. Open the specific group using xarray
                with xr.open_dataset(file_path, group=group_path) as ds:
                    # Let's assume the variable is named 'entrainment' with dimensions (time, z)
                    # Slicing for the current timestep 't'
                    profile = ds["entrainment"].isel(time=t)
                    
                    # 3. Plot the 1D profile (z-axis usually on the vertical y-axis)
                    ax.plot(profile.values, ds["z"].values, label=f"{bot} Entrainment", color='tab:blue', lw=2)
                    
                    # Aesthetics
                    ax.set_xlabel("Entrainment Rate", fontsize=9)
                    ax.set_ylabel("Height z (m)", fontsize=9)
                    ax.grid(True, linestyle="--", alpha=0.5)
                    
            except Exception as e:
                # Handle cases where a specific group might be missing from the file
                ax.text(0.5, 0.5, f"Group not found:\n{group_path}", 
                        ha='center', va='center', transform=ax.transAxes, color='red')
                print(f"⚠️ Error opening group {group_path}: {e}")

            # open group and plot using xarray
            ax.set_title(f"{bot}", fontsize=8, weight='bold')

            while k < bodies:
                sb_body[k].axis('off')
                k += 1
        j += 1

    i += 1

plt.show()